QQQ.
Systems thinking (design) They’re not expecting a perfect system. They want a sane architecture + tradeoffs + scaling story. Framework: “Requirements → API → Data → Scale → Failure → Security → Observability” Requirements (functional + non-functional): QPS, latency, durability, multi-region? API sketch (inputs/outputs) Data model (what’s stored? indexes?) Scale plan (caching, sharding, queues) Failure modes (retries, idempotency, backpressure) Security (authn/z, secrets, least privilege) Observability (SLIs/SLOs, logs/metrics/traces)


Easy Round 1
Design a simple URL shortener for internal company use.


Employees should be able to:
submit a long URL
get back a short link
open the short link and be redirected to the original URL


Keep it simple, but talk through:
requirements
API
data model
scale
failure cases
security
observability

Designing a **simple URL shortener** for internal company use requires a focus on reliability and simplicity over massive global scale. Below is a sane architecture following the requested framework.

### **1. Requirements**
*   **Functional:**
    *   **Shortening:** Submit a long URL and receive a unique short link.
    *   **Redirection:** Clicking the short link redirects the user to the original destination.
*   **Non-Functional:**
    *   **Low Latency:** Redirection must be near-instant (ideally <100ms) to ensure a smooth employee experience.
    *   **Durability:** Links should not "disappear" over time; internal data loss causes lost productivity.
    *   **Availability:** The system must be reliable during business hours.
    *   **Scale:** For internal use, we might expect ~100k URLs total with low QPS (Requests Per Second) compared to public services.

### **2. API Sketch**
The service will follow a **RESTful design**:
*   **POST** `/api/v1/shorten`
    *   **Input:** `{ "longUrl": "string" }`
    *   **Output:** `{ "shortUrl": "https://int.com/aB34" }`.
*   **GET** `/{shortId}`
    *   **Output:** A **302 Redirect** (Temporary). 
    *   *Trade-off:* While a 301 (Permanent) redirect reduces server load, a **302 redirect** is better for internal tools because it allows the company to track click analytics for every single redirect.

### **3. Data Model**
For a simple system, a **relational database** (like PostgreSQL or MySQL) is the best choice because it supports ACID properties (Atomicity, Consistency, Isolation, Durability) out of the box.
*   **Table Structure:**
    *   `id` (BigInt, Primary Key): An auto-incrementing unique ID.
    *   `short_id` (String, Indexed): The Base-62 encoded version of the unique ID.
    *   `long_url` (Text): The original destination.
    *   `created_by` (User ID): For security and auditability.

### **4. Scale Plan**
*   **Caching:** Since redirection is a **read-heavy operation**, we should use an in-memory **Read-Through Cache** (e.g., Redis). When an employee clicks a link, the system first checks the cache; if not found, it queries the DB and populates the cache for future requests.
*   **Load Balancing:** Use a simple **Load Balancer** to distribute traffic between two or more application servers. This prevents a single server from being a **Single Point of Failure (SPOF)**.
*   **Database Scaling:** For internal use, the data will likely fit on one machine. If it grows, we can use **Read Replicas** to handle increased redirection traffic.

### **5. Failure Modes**
*   **Database Down:** If the primary DB fails, the system should fail over to a **Slave/Replica** to maintain availability.
*   **Cache Miss Storm:** If the cache is cleared, the DB might be overwhelmed. We should implement **rate limiting** on the shortening endpoint to prevent script-driven abuse.
*   **Idempotency:** If an employee clicks "shorten" twice for the same URL, the system should recognize the existing entry and return the same short link rather than creating duplicates.

### **6. Security**
*   **Authentication/Authorization:** Integrate with the company's **Single Sign-On (SSO)** to ensure only employees can create or manage links.
*   **Input Validation:** Sanitize long URLs to prevent **SQL injection** or redirections to malicious non-company domains.
*   **Least Privilege:** Ensure the application server has only the minimum necessary permissions to the database.

### **7. Observability**
*   **SLIs/SLOs:** Monitor the **99th percentile latency** for redirects; if it exceeds 200ms, alert the team.
*   **Metrics:** Track the number of 404s (broken links) and 5xx errors (server crashes).
*   **Logging:** Centralize logs to track who created which links, which is vital for **auditability** if a link leads to restricted data.
